In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_visit_detail AS 
WITH encounter_dates_cte AS (
  SELECT
    dbo_encounter.id,
    CASE
      WHEN dbo_encounter.dttm IS NULL THEN NULL
      WHEN CAST(dbo_encounter.dttm AS STRING) = '-' THEN NULL
      WHEN CAST(dbo_encounter.dttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
      ELSE CAST(dbo_encounter.dttm AS TIMESTAMP)
    END AS dttm,
    CASE
      WHEN dbo_visit.startdttm IS NULL THEN NULL
      WHEN CAST(dbo_visit.startdttm AS STRING) = '-' THEN NULL
      WHEN CAST(dbo_visit.startdttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
      ELSE CAST(dbo_visit.startdttm AS TIMESTAMP)
    END AS startdttm,
    CASE
      WHEN dbo_visit.enddttm IS NULL THEN
        CASE
          WHEN dbo_visit.startdttm IS NULL THEN NULL
          WHEN CAST(dbo_visit.startdttm AS STRING) = '-' THEN NULL
          WHEN CAST(dbo_visit.startdttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
          ELSE CAST(dbo_visit.startdttm AS TIMESTAMP)
        END
      WHEN CAST(dbo_visit.enddttm AS STRING) = '-' THEN
        CASE
          WHEN dbo_visit.startdttm IS NULL THEN NULL
          WHEN CAST(dbo_visit.startdttm AS STRING) = '-' THEN NULL
          WHEN CAST(dbo_visit.startdttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
          ELSE CAST(dbo_visit.startdttm AS TIMESTAMP)
        END
      WHEN CAST(dbo_visit.enddttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN
        CASE
          WHEN dbo_visit.startdttm IS NULL THEN NULL
          WHEN CAST(dbo_visit.startdttm AS STRING) = '-' THEN NULL
          WHEN CAST(dbo_visit.startdttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
          ELSE CAST(dbo_visit.startdttm AS TIMESTAMP)
        END
      ELSE CAST(dbo_visit.enddttm AS TIMESTAMP)
    END AS enddttm

  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
    ON dbo_visit.id = dbo_encounter.visitid
)

SELECT
  source_to_person.person_id AS person_id, -- required
  COALESCE(visit_detail_concept.omop_concept_id, 0) AS visit_detail_concept_id, -- required
  CAST(COALESCE(encounter_dates_cte.startdttm, encounter_dates_cte.dttm) AS DATE) AS visit_detail_start_date, -- required
  COALESCE(encounter_dates_cte.startdttm, encounter_dates_cte.dttm) AS visit_detail_start_datetime,
  CAST(COALESCE(encounter_dates_cte.enddttm, encounter_dates_cte.startdttm, encounter_dates_cte.dttm) AS DATE) AS visit_detail_end_date, -- required
  COALESCE(encounter_dates_cte.enddttm, encounter_dates_cte.startdttm, encounter_dates_cte.dttm) AS visit_detail_end_datetime,
  CAST(32817 AS INT) AS visit_detail_type_concept_id, -- required -- 32817 = EHR
  source_to_provider.provider_id AS provider_id,
  COALESCE(billing_care_site.care_site_id, location_care_site.care_site_id) AS care_site_id,
  CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_encounter',
    'id',
    CAST(dbo_encounter.id AS BIGINT)
  ) AS visit_detail_source_value,
  CAST(NULL AS INT) AS visit_detail_source_concept_id,
  CAST(NULL AS INT) AS admitted_from_concept_id,
  CAST(NULL AS STRING) AS admitted_from_source_value,
  CAST(NULL AS INT) AS discharged_to_concept_id,
  CAST(NULL AS STRING) AS discharged_to_source_value,
  CAST(NULL AS BIGINT) AS preceding_visit_detail_id,
  CAST(NULL AS BIGINT) AS parent_visit_detail_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id, -- required
  'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_type_de
  ON dbo_encounter_type_de.id = dbo_encounter.encountertypede
 AND UPPER(dbo_encounter_type_de.isinactiveflag) = 'N'

JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person
  ON dbo_person.id = dbo_encounter.patientid

JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
  ON dbo_visit.id = dbo_encounter.visitid

LEFT JOIN encounter_dates_cte
  ON encounter_dates_cte.id = dbo_encounter.id

JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_person',
    'id',
    CAST(dbo_person.id AS BIGINT)
  )
 AND source_to_person.active_flag = TRUE

JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_visit',
    'id',
    CAST(dbo_visit.id AS BIGINT)
  )
 AND source_to_visit_occurrence.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_provider',
    'id',
    CAST(dbo_encounter.providerid AS BIGINT)
  )
 AND source_to_provider.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_care_site billing_care_site
  ON billing_care_site.care_site_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_billing_location_de',
    'id',
    CAST(NULLIF(CAST(dbo_encounter.defaultbillinglocationde AS BIGINT), 0) AS BIGINT)
  )
 AND billing_care_site.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_care_site location_care_site
  ON location_care_site.care_site_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_location_de',
    'id',
    CAST(NULLIF(CAST(dbo_encounter.primarylocationde AS BIGINT), 0) AS BIGINT)
  )
 AND location_care_site.active_flag = TRUE
JOIN _exponent.omop_mapping.domain_source_to_concept visit_detail_concept
  ON visit_detail_concept.source_system = 'allscripts_tw'
 AND visit_detail_concept.source_table  = 'dbo_encounter_type_de'
 AND LOWER(visit_detail_concept.source_field) = 'entryname'
 AND visit_detail_concept.domain_id     = 'Visit'
 AND visit_detail_concept.source_value  = dbo_encounter_type_de.entryname
 AND visit_detail_concept.active_flag   = TRUE

WHERE dbo_encounter.id IS NOT NULL
  AND visit_detail_concept.omop_concept_id IS NOT NULL
  AND COALESCE(encounter_dates_cte.startdttm, encounter_dates_cte.dttm) IS NOT NULL
  AND source_to_person.person_id IS NOT NULL
  AND source_to_visit_occurrence.visit_occurrence_id IS NOT NULL


In [0]:
%sql
MERGE INTO _exponent.omop_silver.visit_detail AS target
USING silver_visit_detail AS source
ON target.visit_detail_source_value = source.visit_detail_source_value

WHEN MATCHED AND NOT (
     target.person_id                       <=> source.person_id
 AND target.visit_detail_concept_id          <=> source.visit_detail_concept_id
 AND target.visit_detail_start_date          <=> source.visit_detail_start_date
 AND target.visit_detail_start_datetime      <=> source.visit_detail_start_datetime
 AND target.visit_detail_end_date            <=> source.visit_detail_end_date
 AND target.visit_detail_end_datetime        <=> source.visit_detail_end_datetime
 AND target.visit_detail_type_concept_id     <=> source.visit_detail_type_concept_id
 AND target.provider_id                     <=> source.provider_id
 AND target.care_site_id                    <=> source.care_site_id
 AND target.visit_detail_source_value        <=> source.visit_detail_source_value
 AND target.visit_detail_source_concept_id   <=> source.visit_detail_source_concept_id
 AND target.admitted_from_concept_id         <=> source.admitted_from_concept_id
 AND target.admitted_from_source_value       <=> source.admitted_from_source_value
 AND target.discharged_to_concept_id         <=> source.discharged_to_concept_id
 AND target.discharged_to_source_value       <=> source.discharged_to_source_value
 AND target.preceding_visit_detail_id        <=> source.preceding_visit_detail_id
 AND target.parent_visit_detail_id           <=> source.parent_visit_detail_id
 AND target.visit_occurrence_id              <=> source.visit_occurrence_id
 AND target.source_system                    <=> source.source_system
) THEN UPDATE SET
  target.visit_detail_source_value        = source.visit_detail_source_value,
  target.person_id                       = source.person_id,
  target.visit_detail_concept_id          = source.visit_detail_concept_id,
  target.visit_detail_start_date          = source.visit_detail_start_date,
  target.visit_detail_start_datetime      = source.visit_detail_start_datetime,
  target.visit_detail_end_date            = source.visit_detail_end_date,
  target.visit_detail_end_datetime        = source.visit_detail_end_datetime,
  target.visit_detail_type_concept_id     = source.visit_detail_type_concept_id,
  target.provider_id                     = source.provider_id,
  target.care_site_id                    = source.care_site_id,
  target.visit_detail_source_concept_id   = source.visit_detail_source_concept_id,
  target.admitted_from_concept_id         = source.admitted_from_concept_id,
  target.admitted_from_source_value       = source.admitted_from_source_value,
  target.discharged_to_concept_id         = source.discharged_to_concept_id,
  target.discharged_to_source_value       = source.discharged_to_source_value,
  target.preceding_visit_detail_id        = source.preceding_visit_detail_id,
  target.parent_visit_detail_id           = source.parent_visit_detail_id,
  target.visit_occurrence_id              = source.visit_occurrence_id,
  target.source_system                    = source.source_system,
  target.last_mod_tsp                     = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  visit_detail_source_value,
  person_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id,
  visit_occurrence_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.visit_detail_source_value,
  source.person_id,
  source.visit_detail_concept_id,
  source.visit_detail_start_date,
  source.visit_detail_start_datetime,
  source.visit_detail_end_date,
  source.visit_detail_end_datetime,
  source.visit_detail_type_concept_id,
  source.provider_id,
  source.care_site_id,
  source.visit_detail_source_concept_id,
  source.admitted_from_concept_id,
  source.admitted_from_source_value,
  source.discharged_to_concept_id,
  source.discharged_to_source_value,
  source.preceding_visit_detail_id,
  source.parent_visit_detail_id,
  source.visit_occurrence_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_detail (
    source_system,
    visit_detail_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    silver_visit_detail.source_system,
    silver_visit_detail.visit_detail_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(silver_visit_detail.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        visit_detail_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.visit_detail
    WHERE visit_detail_source_value IS NOT NULL
) AS silver_visit_detail
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visit_detail AS existing_visit_detail
  ON silver_visit_detail.visit_detail_source_value = existing_visit_detail.visit_detail_source_value;


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  source_to_visit_detail.visit_detail_id,
  visit_detail.person_id,
  visit_detail.visit_detail_concept_id,
  visit_detail.visit_detail_start_date,
  visit_detail.visit_detail_start_datetime,
  visit_detail.visit_detail_end_date,
  visit_detail.visit_detail_end_datetime,
  visit_detail.visit_detail_type_concept_id,
  visit_detail.provider_id,
  visit_detail.care_site_id,
  visit_detail.visit_detail_source_value,
  visit_detail.visit_detail_source_concept_id,
  visit_detail.admitted_from_concept_id,
  visit_detail.admitted_from_source_value,
  visit_detail.discharged_to_concept_id,
  visit_detail.discharged_to_source_value,
  visit_detail.preceding_visit_detail_id,
  visit_detail.parent_visit_detail_id,
  visit_detail.visit_occurrence_id

FROM _exponent.omop_silver.visit_detail

JOIN _exponent.omop_mapping.source_to_visit_detail
  ON visit_detail.visit_detail_source_value =
     source_to_visit_detail.visit_detail_source_value
 AND source_to_visit_detail.active_flag = TRUE;

In [0]:
%sql
MERGE INTO _exponent.omop.visit_detail AS target
USING gold AS source
ON target.visit_detail_id = source.visit_detail_id

WHEN MATCHED AND NOT (
     target.person_id                    <=> source.person_id
 AND target.visit_detail_concept_id      <=> source.visit_detail_concept_id
 AND target.visit_detail_start_date      <=> source.visit_detail_start_date
 AND target.visit_detail_start_datetime  <=> source.visit_detail_start_datetime
 AND target.visit_detail_end_date        <=> source.visit_detail_end_date
 AND target.visit_detail_end_datetime    <=> source.visit_detail_end_datetime
 AND target.visit_detail_type_concept_id <=> source.visit_detail_type_concept_id
 AND target.provider_id                  <=> source.provider_id
 AND target.care_site_id                 <=> source.care_site_id
 AND target.visit_detail_source_value    <=> source.visit_detail_source_value
 AND target.visit_detail_source_concept_id <=> source.visit_detail_source_concept_id
 AND target.admitted_from_concept_id     <=> source.admitted_from_concept_id
 AND target.admitted_from_source_value   <=> source.admitted_from_source_value
 AND target.discharged_to_concept_id     <=> source.discharged_to_concept_id
 AND target.discharged_to_source_value   <=> source.discharged_to_source_value
 AND target.preceding_visit_detail_id    <=> source.preceding_visit_detail_id
 AND target.parent_visit_detail_id       <=> source.parent_visit_detail_id
 AND target.visit_occurrence_id          <=> source.visit_occurrence_id
) THEN UPDATE SET
  target.person_id                    = source.person_id,
  target.visit_detail_concept_id      = source.visit_detail_concept_id,
  target.visit_detail_start_date      = source.visit_detail_start_date,
  target.visit_detail_start_datetime  = source.visit_detail_start_datetime,
  target.visit_detail_end_date        = source.visit_detail_end_date,
  target.visit_detail_end_datetime    = source.visit_detail_end_datetime,
  target.visit_detail_type_concept_id = source.visit_detail_type_concept_id,
  target.provider_id                  = source.provider_id,
  target.care_site_id                 = source.care_site_id,
  target.visit_detail_source_value    = source.visit_detail_source_value,
  target.visit_detail_source_concept_id = source.visit_detail_source_concept_id,
  target.admitted_from_concept_id     = source.admitted_from_concept_id,
  target.admitted_from_source_value   = source.admitted_from_source_value,
  target.discharged_to_concept_id     = source.discharged_to_concept_id,
  target.discharged_to_source_value   = source.discharged_to_source_value,
  target.preceding_visit_detail_id    = source.preceding_visit_detail_id,
  target.parent_visit_detail_id       = source.parent_visit_detail_id,
  target.visit_occurrence_id          = source.visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_detail_id,
  person_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_value,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id,
  visit_occurrence_id
) VALUES (
  source.visit_detail_id,
  source.person_id,
  source.visit_detail_concept_id,
  source.visit_detail_start_date,
  source.visit_detail_start_datetime,
  source.visit_detail_end_date,
  source.visit_detail_end_datetime,
  source.visit_detail_type_concept_id,
  source.provider_id,
  source.care_site_id,
  source.visit_detail_source_value,
  source.visit_detail_source_concept_id,
  source.admitted_from_concept_id,
  source.admitted_from_source_value,
  source.discharged_to_concept_id,
  source.discharged_to_source_value,
  source.preceding_visit_detail_id,
  source.parent_visit_detail_id,
  source.visit_occurrence_id
);
